# MNIST MLP3 — SGD + momentum baseline

This is a **clean optimizer baseline**. It contains no trace-log projection, adaptive ECS correction, WW-PGD retraction, or spectral-flow intervention.

The notebook runs **three independent complete training seeds**. Every plotted mean is accompanied by a two-sided 95% Student-t confidence interval,

$$
\bar{x} \pm t_{0.975,\,n-1}\frac{s}{\sqrt{n}},
\qquad n=3.
$$

The faint traces are individual seeds. Thick color-coded curves and capped error bars are the mean and 95% confidence interval. The color convention is identical in all three notebooks:

- train: blue;
- test: vermillion;
- FC1: blue;
- FC2: orange;
- FC3: green.

Epoch zero and every training epoch record full train/test cross-entropy and accuracy, including **test accuracy**, plus the original WeightWatcher full-$M$ quantities. The original midpoint is

$$
m_{\mathrm{mid}}
=
\left\lfloor
\frac{m_{\mathrm{detX}}+m_{\mathrm{PL}}}{2}
\right\rfloor .
$$

Optimizer definition: ordinary `torch.optim.SGD` with learning rate 0.05, momentum 0.9, zero dampening, no Nesterov acceleration, and weight decay $10^{-4}$.


In [ ]:
from pathlib import Path
import importlib
import subprocess
import sys

try:
    importlib.import_module("weightwatcher")
except ImportError:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "weightwatcher>=0.7.7"]
    )

ROOT = None
for path in [Path.cwd(), *Path.cwd().parents]:
    candidate = path / "baseline"
    if (candidate / "rg_baselines").is_dir():
        ROOT = candidate
        break
    if (path / "rg_baselines").is_dir():
        ROOT = path
        break
if ROOT is None:
    raise RuntimeError(
        "Run this notebook from a clone of CalculatedContent/rg_optimizers."
    )
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("baseline root:", ROOT)


In [ ]:
from dataclasses import asdict
from IPython.display import display
import pandas as pd

from rg_baselines import (
    BaselineConfig,
    DEFAULT_BASELINE_SEEDS,
    plot_all_replicates,
    run_baseline_replicates,
)

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 1000)

SEEDS = DEFAULT_BASELINE_SEEDS
assert len(SEEDS) == 3 and len(set(SEEDS)) == 3

CONFIG = BaselineConfig(
    optimizer="sgd_momentum",
    epochs=20,
    train_eval_max_batches=None,
    strict_metrics=True,
    sgd_learning_rate=0.05,
    sgd_momentum=0.9,
    sgd_dampening=0.0,
    sgd_nesterov=False,
    sgd_weight_decay=1e-4,
)
RUN_DIR = ROOT / "runs" / "sgd_momentum"
PLOT_DIR = RUN_DIR / "plots"

print("Independent seeds:", SEEDS)
display(pd.DataFrame([asdict(CONFIG)]))


In [ ]:
suite = run_baseline_replicates(
    CONFIG,
    seeds=SEEDS,
    data_dir=ROOT / "data",
    output_dir=RUN_DIR,
    progress=True,
    confidence=0.95,
)

plot_all_replicates(
    suite,
    output_dir=PLOT_DIR,
    show=True,
)

print("saved aggregate results:", RUN_DIR.resolve())
print("saved plots:", PLOT_DIR.resolve())


## Performance per epoch

The table below reports the mean, sample standard deviation, standard error, and two-sided 95% Student-t confidence interval across the three complete runs. `test_accuracy` is shown separately first so the primary outcome cannot be overlooked.


In [ ]:
performance_columns = [
    "epoch", "metric", "n", "mean", "std", "sem",
    "ci_half_width", "ci_low", "ci_high", "minimum", "maximum",
]
test_accuracy_summary = suite.performance_summary.loc[
    suite.performance_summary["metric"].eq("test_accuracy"),
    performance_columns,
].sort_values("epoch")
display(test_accuracy_summary)

required_performance = suite.performance_summary.loc[
    suite.performance_summary["metric"].isin(
        ["train_loss", "test_loss", "train_accuracy", "test_accuracy"]
    ),
    performance_columns,
].sort_values(["metric", "epoch"])
display(required_performance)


## WeightWatcher and midpoint metrics per epoch

These rows aggregate the original WeightWatcher outputs across seeds. `detX_num`, `num_pl_spikes`, and `ERG_gap` come directly from `watcher.analyze(ERG=True)`. The midpoint and trace-log coordinates are then computed from the WeightWatcher-rescaled ESD.


In [ ]:
required_ww_metrics = [
    "alpha",
    "detX_num",
    "num_pl_spikes",
    "ERG_gap",
    "m_midpoint",
    "trace_log_midpoint_per_eval",
    "trace_log_midpoint_total",
]
spectral_columns = [
    "layer", "epoch", "metric", "n", "mean", "std", "sem",
    "ci_half_width", "ci_low", "ci_high", "minimum", "maximum",
]
required_spectral = suite.spectral_summary.loc[
    suite.spectral_summary["metric"].isin(required_ww_metrics),
    spectral_columns,
].sort_values(["metric", "layer", "epoch"])
display(required_spectral)


## Additional per-epoch diagnostics

The following table includes effective ranks, retained-energy fractions, spectral conditioning, normalization audits, gradient norms, parameter norms, and timing. All aggregate rows retain the same seed-level 95% confidence-interval definition.


In [ ]:
additional_spectral_metrics = [
    "stable_rank",
    "participation_ratio",
    "entropy_effective_rank",
    "boundary_overlap_ratio",
    "top1_energy_fraction",
    "pl_energy_fraction",
    "detx_energy_fraction",
    "midpoint_energy_fraction",
    "geometric_mean_midpoint",
    "normalized_lambda_max",
    "normalized_lambda_midpoint_cut",
    "eigenvalue_condition_number",
]
display(
    suite.spectral_summary.loc[
        suite.spectral_summary["metric"].isin(additional_spectral_metrics),
        spectral_columns,
    ].sort_values(["metric", "layer", "epoch"])
)

display(
    suite.performance_summary.loc[
        suite.performance_summary["metric"].isin(
            [
                "mean_gradient_norm_before_clip",
                "max_gradient_norm_before_clip",
                "parameter_l2_norm",
                "train_time_sec",
                "evaluation_time_sec",
                "weightwatcher_time_sec",
            ]
        ),
        performance_columns,
    ].sort_values(["metric", "epoch"])
)


In [ ]:
# Final audit: all required seed/epoch/layer measurements are present.
expected_epochs = set(range(CONFIG.epochs + 1))
assert set(suite.performance["epoch"].astype(int)) == expected_epochs
assert set(suite.performance["seed"].astype(int)) == set(SEEDS)

valid = suite.spectral_metrics.loc[suite.spectral_metrics["status"].eq("ok")]
for epoch in expected_epochs:
    for layer in ("fc1", "fc2", "fc3"):
        observed = set(
            valid.loc[
                valid["epoch"].eq(epoch) & valid["layer"].eq(layer),
                "seed",
            ].astype(int)
        )
        assert observed == set(SEEDS), (epoch, layer, observed)

print(
    "Audit passed:",
    len(SEEDS),
    "independent seeds;",
    CONFIG.epochs + 1,
    "epochs including epoch 0;",
    "FC1/FC2/FC3 WeightWatcher metrics at every epoch.",
)
